<a href="https://colab.research.google.com/github/singamsrikanth77-spec/MyTrade/blob/main/mytrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/kv_files/scanner.kv
#:import dp kivy.metrics.dp

<ScannerScreen>:
    MDBoxLayout:
        orientation: "vertical"
        padding: dp(10)
        spacing: dp(10)

        MDTopAppBar:
            title: "MyTradeAlert - Scanner"
            elevation: 1
            left_action_items: [["chart-line", lambda x: None]]

        MDCard:
            size_hint_y: None
            height: dp(52)
            padding: dp(8)
            radius: [16, 16, 16, 16]
            md_bg_color: 0.12, 0.12, 0.14, 1

            MDBoxLayout:
                orientation: "horizontal"
                spacing: dp(10)

                MDLabel:
                    text: "Symbol | TF | Signal | Price | Score | Phase"
                    bold: True
                    valign: "middle"
                    halign: "left"

                MDSwitch:
                    id: active_only_switch
                    active: False
                    pos_hint: {"center_y": 0.5}

        ScrollView:
            do_scroll_x: False

            MDBoxLayout:
                id: scanner_list
                orientation: "vertical"
                size_hint_y: None
                height: self.minimum_height
                spacing: dp(10)
                padding: 0, dp(2), 0, dp(8)

In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/screens/scanner_screen.py

import threading
from kivy.clock import Clock
from kivy.metrics import dp
from kivymd.app import MDApp
from kivymd.uix.screen import MDScreen
from kivymd.uix.card import MDCard
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel, MDIcon

class ScannerScreen(MDScreen):
    def refresh_scanner(self, results, active_only=False):
        self.ids.scanner_list.clear_widgets()
        for item in reversed(results[-80:]):
            signal = item.get("signal", "None")
            if active_only and signal == "None":
                continue

            if signal in ("Buy Call", "Analyse 21 Bullish"):
                bg = (0.12, 0.18, 0.14, 1)
                icon = "arrow-up-bold"
                icon_color = (0.35, 0.9, 0.45, 1)
            elif signal in ("Buy Put", "Analyse 21 Bearish"):
                bg = (0.18, 0.12, 0.12, 1)
                icon = "arrow-down-bold"
                icon_color = (0.95, 0.35, 0.35, 1)
            else:
                bg = (0.12, 0.12, 0.12, 1)
                icon = "chart-line"
                icon_color = (0.7, 0.7, 0.7, 1)

            self.ids.scanner_list.add_widget(
                self.make_signal_card(
                    symbol=item.get("symbol", ""),
                    tf=item.get("tf", ""),
                    signal=signal,
                    price=item.get("price", ""),
                    score=item.get("score", 0),
                    confirmation_score=item.get("confirmation_score", 0),
                    phase=item.get("phase", "unknown"),
                    time_text=item.get("time", ""),
                    hits=item.get("hits", []),
                    reason=item.get("reason", ""),
                    bg=bg,
                    icon=icon,
                    icon_color=icon_color,
                )
            )

    def make_signal_card(self, symbol, tf, signal, price, score, confirmation_score, phase, time_text, hits, reason, bg, icon, icon_color):
        card = MDCard(
            size_hint_y=None,
            height=dp(128),
            padding=dp(12),
            radius=[16, 16, 16, 16],
            md_bg_color=bg,
        )

        row = MDBoxLayout(orientation="horizontal", spacing=dp(10))
        row.add_widget(
            MDIcon(
                icon=icon,
                theme_text_color="Custom",
                text_color=icon_color,
                size_hint_x=None,
                width=dp(28),
            )
        )

        col = MDBoxLayout(orientation="vertical", spacing=dp(3))
        col.add_widget(MDLabel(text=f"{symbol} | {tf}", bold=True))
        col.add_widget(MDLabel(text=f"Signal: {signal}   Price: {price}   Score: {score}"))
        col.add_widget(MDLabel(text=f"AI Confirm: {confirmation_score}   Phase: {phase}"))
        col.add_widget(MDLabel(text=f"Hits: {','.join(map(str, hits)) or '-'}"))
        col.add_widget(MDLabel(text=f"Reason: {reason}"))
        col.add_widget(MDLabel(text=f"Time: {time_text}"))

        row.add_widget(col)
        card.add_widget(row)
        return card

    def on_toggle_active_only(self, active):
        app = MDApp.get_running_app()
        if hasattr(app, "ai"):
            self.refresh_scanner(app.ai.latest_scan_results, active)

    def on_pre_enter(self):
        app = MDApp.get_running_app()
        if hasattr(app, "ai"):
            self.refresh_scanner(app.ai.latest_scan_results, self.ids.active_only_switch.active)

In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/kv_files/watchlist.kv
#:import dp kivy.metrics.dp

<WatchlistScreen>:
    MDBoxLayout:
        orientation: "vertical"
        padding: dp(10)
        spacing: dp(10)

        MDTopAppBar:
            title: "MyTradeAlert - Watchlist"
            elevation: 1
            left_action_items: [["heart", lambda x: None]]

        MDCard:
            size_hint_y: None
            height: dp(60)
            padding: dp(8)
            radius: [16, 16, 16, 16]
            md_bg_color: 0.12, 0.12, 0.14, 1

            MDBoxLayout:
                spacing: dp(8)

                MDTextField:
                    id: search_field
                    hint_text: "Search ticker"
                    mode: "rectangle"
                    on_text: root.on_search_text(self.text)

                MDRaisedButton:
                    text: "[+] Add"
                    on_release: root.add_selected_symbol()

                MDIconButton:
                    icon: "close"
                    on_release: root.clear_search()

        ScrollView:
            do_scroll_x: False

            MDBoxLayout:
                id: watchlist_content
                orientation: "vertical"
                size_hint_y: None
                height: self.minimum_height
                spacing: dp(10)
                padding: 0, dp(2), 0, dp(8)

        MDCard:
            size_hint_y: None
            height: dp(150)
            padding: dp(8)
            radius: [16, 16, 16, 16]
            md_bg_color: 0.10, 0.10, 0.12, 1

            ScrollView:
                do_scroll_x: False

                MDBoxLayout:
                    id: suggestions_box
                    orientation: "vertical"
                    size_hint_y: None
                    height: self.minimum_height
                    spacing: dp(6)

In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/screens/watchlist_screen.py

import threading
from kivy.clock import Clock
from kivy.metrics import dp
from kivymd.app import MDApp
from kivymd.uix.screen import MDScreen
from kivymd.uix.card import MDCard
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel, MDIcon
from kivymd.uix.button import MDIconButton

class WatchlistScreen(MDScreen):
    def refresh_watchlist(self, watchlist, states):
        self.ids.watchlist_content.clear_widgets()
        for symbol in watchlist:
            state = states.get(symbol, {"signal": "None", "tf": "", "price": 0, "score": 0, "confirmation_score": 0, "phase": "unknown", "hits": [], "reason": ""})
            signal = state.get("signal", "None")

            if signal in ("Buy Call", "Analyse 21 Bullish"):
                bg = (0.12, 0.18, 0.14, 1)
                icon = "arrow-up-bold"
                icon_color = (0.35, 0.9, 0.45, 1)
            elif signal in ("Buy Put", "Analyse 21 Bearish"):
                bg = (0.18, 0.12, 0.12, 1)
                icon = "arrow-down-bold"
                icon_color = (0.95, 0.35, 0.35, 1)
            else:
                bg = (0.11, 0.11, 0.11, 1)
                icon = "eye-outline"
                icon_color = (0.7, 0.7, 0.7, 1)

            self.ids.watchlist_content.add_widget(
                self.make_watch_card(symbol, state, bg, icon, icon_color)
            )

    def make_watch_card(self, symbol, state, bg, icon, icon_color):
        card = MDCard(
            size_hint_y=None,
            height=dp(190),
            padding=dp(12),
            radius=[16, 16, 16, 16],
            md_bg_color=bg,
        )

        row = MDBoxLayout(orientation="horizontal", spacing=dp(10))
        row.add_widget(
            MDIcon(
                icon=icon,
                theme_text_color="Custom",
                text_color=icon_color,
                size_hint_x=None,
                width=dp(28),
            )
        )

        col = MDBoxLayout(orientation="vertical", spacing=dp(4))
        col.add_widget(MDLabel(text=symbol, bold=True))
        col.add_widget(MDLabel(text=f"Signal: {state.get('signal', 'None')}"))
        col.add_widget(MDLabel(text=f"TF: {state.get('tf', '')}   Price: {state.get('price', 0)}"))
        col.add_widget(MDLabel(text=f"Score: {state.get('score', 0)}   AI Confirm: {state.get('confirmation_score', 0)}"))
        col.add_widget(MDLabel(text=f"Hits: {','.join(map(str, state.get('hits', []))) or '-'}"))
        col.add_widget(MDLabel(text=f"Reason: {state.get('reason', '')}"))
        col.add_widget(MDLabel(text=f"Phase: {state.get('phase', 'unknown')}"))

        row.add_widget(col)
        card.add_widget(row)
        return card

    def on_search_text(self, text):
        app = MDApp.get_running_app()
        self.ids.suggestions_box.clear_widgets()
        if len(text.strip()) < 2:
            return

        def worker():
            import asyncio
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            results = loop.run_until_complete(app.stock_search.autocomplete(text.strip()))
            Clock.schedule_once(lambda dt: self._show_suggestions(results))

        threading.Thread(target=worker, daemon=True).start()

    def _show_suggestions(self, results):
        self.ids.suggestions_box.clear_widgets()
        for r in results[:12]:
            row = MDBoxLayout(size_hint_y=None, height=dp(42), spacing=dp(8))
            row.add_widget(MDLabel(text=r))
            btn = MDIconButton(icon="plus-circle")
            btn.bind(on_release=lambda inst, val=r: self._pick_suggestion(val))
            row.add_widget(btn)
            self.ids.suggestions_box.add_widget(row)

    def _pick_suggestion(self, value):
        self.ids.search_field.text = value.split(" - ")[0]
        self.ids.suggestions_box.clear_widgets()

    def add_selected_symbol(self):
        app = MDApp.get_running_app()
        text = self.ids.search_field.text.strip()
        if not text:
            return

        def worker():
            import asyncio
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            info = loop.run_until_complete(app.stock_search.validate_ticker(text))
            if info:
                if text not in app.watchlist:
                    app.watchlist.append(text)
                    # Assuming storage is available via app instance
                    if hasattr(app, 'storage'):
                        app.storage.put("watchlist", value=app.watchlist)
                    Clock.schedule_once(lambda dt: app.show_snackbar(f"Added {text}"))
                    Clock.schedule_once(lambda dt: self.refresh_watchlist(app.watchlist, app.ai.latest_symbol_states))
            else:
                Clock.schedule_once(lambda dt: app.show_snackbar("Ticker data unavailable"))

        threading.Thread(target=worker, daemon=True).start()

    def clear_search(self):
        self.ids.search_field.text = ""
        self.ids.suggestions_box.clear_widgets()

    def on_pre_enter(self):
        app = MDApp.get_running_app()
        if hasattr(app, "ai"):
            self.refresh_watchlist(app.watchlist, app.ai.latest_symbol_states)

Overwriting /content/drive/MyDrive/MyTrade/screens/watchlist_screen.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/kv_files/backtest.kv
#:import dp kivy.metrics.dp

<BacktestScreen>:
    MDBoxLayout:
        orientation: "vertical"
        padding: dp(10)
        spacing: dp(10)

        MDTopAppBar:
            title: "MyTradeAlert - Backtest"
            elevation: 1
            left_action_items: [["clipboard-text", lambda x: None]]

        MDCard:
            size_hint_y: None
            height: dp(60)
            padding: dp(8)
            radius: [16, 16, 16, 16]
            md_bg_color: 0.12, 0.12, 0.14, 1

            MDBoxLayout:
                spacing: dp(8)

                MDTextField:
                    id: backtest_search
                    hint_text: "Search asset"
                    mode: "rectangle"
                    on_text: root.on_search_text(self.text)

                MDRaisedButton:
                    text: "Run Backtest"
                    on_release: root.run_selected_backtest()

                MDIconButton:
                    icon: "close"
                    on_release: root.clear_search()

        ScrollView:
            do_scroll_x: False

            MDBoxLayout:
                id: backtest_content
                orientation: "vertical"
                size_hint_y: None
                height: self.minimum_height
                spacing: dp(10)
                padding: 0, dp(2), 0, dp(8)

                MDCard:
                    size_hint_y: None
                    height: dp(150)
                    padding: dp(8)
                    radius: [16, 16, 16, 16]
                    md_bg_color: 0.10, 0.10, 0.12, 1

                    ScrollView:
                        do_scroll_x: False

                        MDBoxLayout:
                            id: backtest_suggestions
                            orientation: "vertical"
                            size_hint_y: None
                            height: self.minimum_height
                            spacing: dp(6)

Overwriting /content/drive/MyDrive/MyTrade/kv_files/backtest.kv


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/screens/backtest_screen.py
import threading
from kivy.clock import Clock
from kivy.metrics import dp
from kivymd.app import MDApp
from kivymd.uix.screen import MDScreen
from kivymd.uix.card import MDCard
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel, MDIcon
from kivymd.uix.button import MDIconButton

class BacktestScreen(MDScreen):
    def refresh_backtest_results(self, results):
        self.ids.backtest_content.clear_widgets()
        for r in results[-20:]:
            self.ids.backtest_content.add_widget(self.make_backtest_card(r))

    def make_backtest_card(self, r):
        card = MDCard(
            size_hint_y=None,
            height=dp(160),
            padding=dp(12),
            radius=[16, 16, 16, 16],
            md_bg_color=(0.12, 0.12, 0.12, 1),
        )

        row = MDBoxLayout(orientation="horizontal", spacing=dp(10))
        row.add_widget(
            MDIcon(
                icon="clipboard-text",
                theme_text_color="Custom",
                text_color=(0.6, 0.7, 1, 1),
                size_hint_x=None,
                width=dp(28),
            )
        )

        col = MDBoxLayout(orientation="vertical", spacing=dp(4))
        col.add_widget(MDLabel(text=f"{r.get('symbol', '')} | {r.get('strategy', 'Analyse 21')}", bold=True))
        col.add_widget(MDLabel(text=f"Profit: {r.get('profit', 0)}%   Winrate: {r.get('winrate', 0)}%   Loss: {r.get('loss', 0)}%"))
        col.add_widget(MDLabel(text=f"Trades: {r.get('total_trades', 0)}   DD: {r.get('max_drawdown', 0)}   Score: {r.get('score', 0)}"))
        col.add_widget(MDLabel(text=f"AI Confirm: {r.get('confirmation_score', 0)}   Phase: {r.get('phase', 'unknown')}"))

        row.add_widget(col)
        card.add_widget(row)
        return card

    def on_search_text(self, text):
        app = MDApp.get_running_app()
        self.ids.backtest_suggestions.clear_widgets()
        if len(text.strip()) < 2:
            return

        def worker():
            import asyncio
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            results = loop.run_until_complete(app.stock_search.autocomplete(text.strip()))
            Clock.schedule_once(lambda dt: self._show_suggestions(results))

        threading.Thread(target=worker, daemon=True).start()

    def _show_suggestions(self, results):
        self.ids.backtest_suggestions.clear_widgets()
        for r in results[:12]:
            row = MDBoxLayout(size_hint_y=None, height=dp(42), spacing=dp(8))
            row.add_widget(MDLabel(text=r))
            btn = MDIconButton(icon="plus-circle")
            btn.bind(on_release=lambda inst, val=r: self._pick_suggestion(val))
            row.add_widget(btn)
            self.ids.backtest_suggestions.add_widget(row)

    def _pick_suggestion(self, value):
        self.ids.backtest_search.text = value.split(" - ")[0]
        self.ids.backtest_suggestions.clear_widgets()

    def clear_search(self):
        self.ids.backtest_search.text = ""
        self.ids.backtest_suggestions.clear_widgets()

    def run_selected_backtest(self):
        app = MDApp.get_running_app()
        symbol = self.ids.backtest_search.text.strip()
        if not symbol:
            return

        def worker():
            import asyncio
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            result = loop.run_until_complete(app.backtester.backtest_symbol(symbol))
            app.backtester.latest_results = [result]
            app.ai.memory.update_from_backtest([result])
            Clock.schedule_once(lambda dt: self.refresh_backtest_results(app.backtester.latest_results))
            Clock.schedule_once(lambda dt: app.show_snackbar(f"Backtest completed for {symbol}"))

        threading.Thread(target=worker, daemon=True).start()

    def on_pre_enter(self):
        app = MDApp.get_running_app()
        if hasattr(app, "backtester"):
            self.refresh_backtest_results(app.backtester.latest_results)

Overwriting /content/drive/MyDrive/MyTrade/screens/backtest_screen.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/kv_files/settings.kv
#:import dp kivy.metrics.dp

<SettingsScreen>:
    MDBoxLayout:
        orientation: "vertical"
        padding: dp(10)
        spacing: dp(10)

        MDTopAppBar:
            title: "MyTradeAlert - Settings"
            elevation: 1
            left_action_items: [["cog", lambda x: None]]

        ScrollView:
            do_scroll_x: False

            MDBoxLayout:
                id: settings_content
                orientation: "vertical"
                size_hint_y: None
                height: self.minimum_height
                spacing: dp(10)
                padding: 0, dp(2), 0, dp(8)

                MDCard:
                    size_hint_y: None
                    height: dp(245)
                    padding: dp(12)
                    radius: [16, 16, 16, 16]
                    md_bg_color: 0.10, 0.10, 0.12, 1

                    MDBoxLayout:
                        orientation: "vertical"
                        spacing: dp(8)

                        MDLabel:
                            text: "Telegram Settings"
                            bold: True

                        MDTextField:
                            id: bot_token
                            hint_text: "Bot Token"
                            password: True
                            mode: "rectangle"
                            on_text: root.clear_status()

                        MDTextField:
                            id: chat_id
                            hint_text: "Chat ID"
                            mode: "rectangle"
                            input_filter: "int"
                            on_text: root.clear_status()

                        MDBoxLayout:
                            size_hint_y: None
                            height: dp(48)
                            spacing: dp(8)

                            MDRaisedButton:
                                text: "Test Connection"
                                on_release: root.test_connection()

                            MDRaisedButton:
                                text: "Send Test Alert"
                                on_release: root.send_test_alert()

                MDCard:
                    size_hint_y: None
                    height: dp(185)
                    padding: dp(12)
                    radius: [16, 16, 16, 16]
                    md_bg_color: 0.10, 0.10, 0.12, 1

                    MDBoxLayout:
                        orientation: "vertical"
                        spacing: dp(6)

                        MDLabel:
                            text: "AI & Alerts"
                            bold: True

                        MDBoxLayout:
                            adaptive_height: True
                            spacing: dp(8)

                            MDLabel:
                                text: "Enable Telegram Alerts"

                            MDSwitch:
                                id: enable_telegram
                                disabled: True
                                on_active: root.toggle_telegram(self.active)

                        MDBoxLayout:
                            adaptive_height: True
                            spacing: dp(8)

                            MDLabel:
                                text: "Enable AI Analysis"

                            MDSwitch:
                                id: enable_ai
                                on_active: root.toggle_ai(self.active)

                        MDLabel:
                            id: ai_status_label
                            text: "AI Status: Stopped"

                        MDLabel:
                            id: last_error
                            text: "Last Error: None"
                            theme_text_color: "Custom"
                            text_color: 1, 0.4, 0.4, 1

Overwriting /content/drive/MyDrive/MyTrade/kv_files/settings.kv


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/screens/settings_screen.py

import threading
from kivy.clock import Clock
from kivymd.app import MDApp
from kivymd.uix.screen import MDScreen

class SettingsScreen(MDScreen):
    def clear_status(self):
        self.ids.last_error.text = "Last Error: None"

    def refresh_status(self, ai_status, last_error):
        self.ids.ai_status_label.text = ai_status
        self.ids.last_error.text = f"Last Error: {last_error or 'None'}"

    def toggle_telegram(self, value):
        app = MDApp.get_running_app()
        app.telegram_enabled = value
        app._save_storage("telegram_enabled", value)

    def toggle_ai(self, value):
        app = MDApp.get_running_app()
        app.ai_enabled = value
        app._save_storage("ai_enabled", value)
        if value and not app.ai.running:
            threading.Thread(target=app.ai.run_forever, daemon=True).start()
        elif not value:
            app.ai.stop()


    def test_connection(self):
        app = MDApp.get_running_app()
        token = self.ids.bot_token.text.strip()
        chat_id = self.ids.chat_id.text.strip()

        def worker():
            import asyncio
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            ok, msg = loop.run_until_complete(app.telegram.test_connection(token, chat_id))

            def done(dt):
                if ok:
                    app.telegram_enabled = True
                    app._save_storage("tg_token", token)
                    app._save_storage("tg_chat_id", chat_id)
                    app._save_storage("telegram_enabled", True)
                    self.ids.enable_telegram.disabled = False
                    self.ids.enable_telegram.active = True
                    app.show_snackbar("Telegram connected")
                else:
                    self.ids.last_error.text = f"Last Error: {msg}"
                    app.show_snackbar(msg)

            Clock.schedule_once(done)

        threading.Thread(target=worker, daemon=True).start()

    def send_test_alert(self):
        app = MDApp.get_running_app()
        token = self.ids.bot_token.text.strip()
        chat_id = self.ids.chat_id.text.strip()

        def worker():
            import asyncio
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            result = loop.run_until_complete(
                app.telegram.send_message(token, chat_id, "MyTradeAlert: Test ping successful")
            )
            Clock.schedule_once(
                lambda dt: app.show_snackbar(
                    "Test alert sent" if result.get("ok") else result.get("description", "Failed")
                )
            )

        threading.Thread(target=worker, daemon=True).start()

    def on_pre_enter(self):
        app = MDApp.get_running_app()
        self.ids.bot_token.text = app._load_storage("tg_token", "")
        self.ids.chat_id.text = app._load_storage("tg_chat_id", "")
        self.ids.enable_telegram.active = app.telegram_enabled
        self.ids.enable_ai.active = app.ai_enabled
        self.refresh_status(
            getattr(app.ai, "get_status_text", lambda: "AI Status: Stopped")(),
            getattr(app.ai, "last_error", "")
        )

Overwriting /content/drive/MyDrive/MyTrade/screens/settings_screen.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/kv_files/main.kv
<MDNavigationLayout>:
    MDScreenManager:
        id: screen_manager

        ScannerScreen:
            name: 'scanner'
        WatchlistScreen:
            name: 'watchlist'
        BacktestScreen:
            name: 'backtest'
        SettingsScreen:
            name: 'settings'

    MDNavigationDrawer:
        id: nav_drawer
        radius: (0, 16, 16, 0)

        MDNavigationDrawerMenu:
            MDNavigationDrawerHeader:
                title: "MyTrade"
                spacing: "4dp"
                padding: "12dp", 0, 0, "12dp"

            MDNavigationDrawerItem:
                icon: "chart-line"
                text: "Scanner"
                on_release:
                    app.switch_screen('scanner')
                    nav_drawer.set_state('close')

            MDNavigationDrawerItem:
                icon: "heart"
                text: "Watchlist"
                on_release:
                    app.switch_screen('watchlist')
                    nav_drawer.set_state('close')

            MDNavigationDrawerItem:
                icon: "clipboard-text"
                text: "Backtest"
                on_release:
                    app.switch_screen('backtest')
                    nav_drawer.set_state('close')

            MDNavigationDrawerItem:
                icon: "cog"
                text: "Settings"
                on_release:
                    app.switch_screen('settings')
                    nav_drawer.set_state('close')

Overwriting /content/drive/MyDrive/MyTrade/kv_files/main.kv


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/main.py
import json
import threading
import os

# Set environment variables to avoid Kivy Window issues in headless environment
os.environ['KIVY_NO_ARGS'] = '1'
os.environ['KIVY_WINDOW'] = 'sdl2'

from kivy.clock import Clock
from kivy.lang import Builder
from kivymd.app import MDApp
from kivymd.uix.snackbar import MDSnackbar
# Fixed: Using MDRaisedButton for KivyMD 1.2.0 compatibility
from kivymd.uix.button import MDRaisedButton

from screens.scanner_screen import ScannerScreen
from screens.watchlist_screen import WatchlistScreen
from screens.backtest_screen import BacktestScreen
from screens.settings_screen import SettingsScreen

from modules.ai_trend_learner import AITrendLearner
from modules.backtester import Backtester
from modules.telegram_alerts import TelegramAlerts
from modules.storage import AppStorage

from utils.data_fetcher import DataFetcher
from utils.stock_search import StockSearch
from utils.memory_manager import MemoryManager

class MyTradeAlertApp(MDApp):
    def build(self):
        self.title = "MyTradeAlert"
        self.theme_cls.theme_style = "Dark"
        self.theme_cls.primary_palette = "BlueGray"

        self.storage = AppStorage("data/persisted/app_store.json")
        self.watchlist = []
        self.telegram_enabled = False
        self.ai_enabled = False
        self.last_error = ""

        self.data_fetcher = DataFetcher(app=self)
        self.stock_search = StockSearch(app=self)
        self.telegram = TelegramAlerts(app=self)
        self.ai = AITrendLearner(app=self, data_fetcher=self.data_fetcher, telegram=self.telegram)
        self.backtester = Backtester(app=self, data_fetcher=self.data_fetcher, ai=self.ai)

        Builder.load_file("kv_files/scanner.kv")
        Builder.load_file("kv_files/watchlist.kv")
        Builder.load_file("kv_files/backtest.kv")
        Builder.load_file("kv_files/settings.kv")
        return Builder.load_file("kv_files/main.kv")

    def on_start(self):
        print("Application started. Initializing background tasks...")
        if self.ai_enabled:
            threading.Thread(target=self.ai.run_forever, daemon=True).start()
        Clock.schedule_interval(self._ui_refresh_tick, 10)

    def show_snackbar(self, text):
        # Simplified for 1.2.0
        MDSnackbar(text=text).open()

    def switch_screen(self, name):
        self.root.ids.screen_manager.current = name

    def _ui_refresh_tick(self, dt):
        pass

if __name__ == "__main__":
    MyTradeAlertApp().run()

Overwriting /content/drive/MyDrive/MyTrade/main.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/analysis/technical_analysis.py

import pandas as pd

try:
    import pandas_ta as ta
except Exception:
    ta = None

class TechnicalAnalysis:
    @staticmethod
    def _normalize_df(df):
        if df is None or df.empty:
            return None
        df = df.copy()
        df.columns = [str(c).title() for c in df.columns]
        needed = {"Open", "High", "Low", "Close", "Volume"}
        if not needed.issubset(df.columns):
            return None
        return df.dropna().copy()

    @staticmethod
    def _manual_bbands(close, length=20, std=2.0):
        mid = close.rolling(length).mean()
        dev = close.rolling(length).std(ddof=0)
        lower = mid - std * dev
        upper = mid + std * dev
        return lower, mid, upper

    @staticmethod
    def _manual_rsi(close, length=14):
        delta = close.diff()
        gain = delta.clip(lower=0)
        loss = (-delta).clip(lower=0)
        avg_gain = gain.ewm(alpha=1 / length, adjust=False).mean()
        avg_loss = loss.ewm(alpha=1 / length, adjust=False).mean()
        rs = avg_gain / avg_loss.replace(0, pd.NA)
        rsi = 100 - (100 / (1 + rs))
        return rsi.bfill()

    @staticmethod
    def add_indicators(df, bb_length=20, bb_std=2.0, rsi_length=14):
        df = TechnicalAnalysis._normalize_df(df)
        if df is None:
            return None

        if ta is not None:
            try:
                bb = ta.bbands(df["Close"], length=bb_length, std=bb_std)
                if bb is not None and not bb.empty:
                    df["bb_lower"] = bb.iloc[:, 0]
                    df["bb_mid"] = bb.iloc[:, 1]
                    df["bb_upper"] = bb.iloc[:, 2]
                else:
                    raise ValueError("empty bbands")
            except Exception:
                lower, mid, upper = TechnicalAnalysis._manual_bbands(df["Close"], bb_length, bb_std)
                df["bb_lower"], df["bb_mid"], df["bb_upper"] = lower, mid, upper

            try:
                df["rsi"] = ta.rsi(df["Close"], length=rsi_length)
            except Exception:
                df["rsi"] = TechnicalAnalysis._manual_rsi(df["Close"], rsi_length)
        else:
            lower, mid, upper = TechnicalAnalysis._manual_bbands(df["Close"], bb_length, bb_std)
            df["bb_lower"], df["bb_mid"], df["bb_upper"] = lower, mid, upper
            df["rsi" ] = TechnicalAnalysis._manual_rsi(df["Close"], rsi_length)

        return df.dropna().copy()

    @staticmethod
    def bb_lower_cross(df):
        if df is None or len(df) < 2:
            return False
        prev = df.iloc[-2]
        last = df.iloc[-1]
        return bool(prev["Close"] <= prev["bb_lower"] and last["Close"] > last["bb_lower"])

    @staticmethod
    def bb_upper_cross(df):
        if df is None or len(df) < 2:
            return False
        prev = df.iloc[-2]
        last = df.iloc[-1]
        return bool(prev["Close"] >= prev["bb_upper"] and last["Close"] < last["bb_upper"])

    @staticmethod
    def rsi_bullish_divergence(df, lookback=14):
        if df is None or len(df) < lookback + 2:
            return False
        w = df.tail(lookback)
        return bool(w["Low"].iloc[-1] <= w["Low"].min() and w["rsi"].iloc[-1] > w["rsi"].iloc[-3])

    @staticmethod
    def rsi_bearish_divergence(df, lookback=14):
        if df is None or len(df) < lookback + 2:
            return False
        w = df.tail(lookback)
        return bool(w["High"].iloc[-1] >= w["High"].max() and w["rsi"].iloc[-1] < w["rsi"].iloc[-3])

    @staticmethod
    def market_phase(df):
        if df is None or len(df) < 30:
            return "unknown"
        last = df.iloc[-1]
        price = float(last["Close"])
        upper = float(last["bb_upper"])
        lower = float(last["bb_lower"])
        rsi = float(last["rsi"])
        if price > upper and rsi > 60:
            return "distribution"
        if price < lower and rsi < 40:
            return "accumulation"
        return "consolidation"

Overwriting /content/drive/MyDrive/MyTrade/analysis/technical_analysis.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/analysis/rule_engine.py

class RuleEngine:
    @staticmethod
    def candle_stats(row):
        o = float(row["Open"])
        h = float(row["High"])
        l = float(row["Low"])
        c = float(row["Close"])
        body = abs(c - o)
        rng = max(h - l, 1e-9)
        upper_wick = h - max(o, c)
        lower_wick = min(o, c) - l
        return {
            "body_ratio": body / rng,
            "upper_wick_ratio": max(upper_wick, 0.0) / rng,
            "lower_wick_ratio": max(lower_wick, 0.0) / rng,
            "bull": c >= o,
            "bear": c < o,
        }

    @staticmethod
    def bollinger_squeeze(df, lookback=20, compression=0.8):
        if len(df) < lookback + 2:
            return False
        width = (df["bb_upper"] - df["bb_lower"]).tail(lookback)
        return bool(width.iloc[-1] <= width.mean() * compression)

    @staticmethod
    def volume_surge(df, lookback=20, mult=1.5):
        if len(df) < lookback + 2:
            return False
        base = df["Volume"].tail(lookback)
        return bool(df["Volume"].iloc[-1] >= base.mean() * mult)

    @staticmethod
    def trend_slope(df, lookback=10):
        if len(df) < lookback + 2:
            return 0
        close = df["Close"].tail(lookback)
        slope = close.iloc[-1] - close.iloc[0]
        return 1 if slope > 0 else (-1 if slope < 0 else 0)

    @staticmethod
    def bullish_divergence(df, lookback=14):
        if len(df) < lookback + 2:
            return False
        w = df.tail(lookback)
        return bool(w["Low"].iloc[-1] <= w["Low"].min() and w["rsi"].iloc[-1] > w["rsi"].iloc[-3])

    @staticmethod
    def bearish_divergence(df, lookback=14):
        if len(df) < lookback + 2:
            return False
        w = df.tail(lookback)
        return bool(w["High"].iloc[-1] >= w["High"].max() and w["rsi"].iloc[-1] < w["rsi"].iloc[-3])

    @staticmethod
    def support_rejection(df):
        if len(df) < 3:
            return False
        last = df.iloc[-1]
        prev = df.iloc[-2]
        stats = RuleEngine.candle_stats(last)
        return bool(prev["Close"] <= prev["bb_lower"] and last["Close"] > last["Open"] and stats["lower_wick_ratio"] > stats["upper_wick_ratio"])

    @staticmethod
    def resistance_rejection(df):
        if len(df) < 3:
            return False
        last = df.iloc[-1]
        prev = df.iloc[-2]
        stats = RuleEngine.candle_stats(last)
        return bool(prev["Close"] >= prev["bb_upper"] and last["Close"] < last["Open"] and stats["upper_wick_ratio"] > stats["lower_wick_ratio"])

    @staticmethod
    def breakout_up(df):
        if len(df) < 3:
            return False
        prev = df.iloc[-2]
        last = df.iloc[-1]
        return bool(prev["Close"] <= prev["bb_mid"] and last["Close"] > last["bb_mid"] and last["Close"] > prev["Close"])

    @staticmethod
    def breakout_down(df):
        if len(df) < 3:
            return False
        prev = df.iloc[-2]
        last = df.iloc[-1]
        return bool(prev["Close"] >= prev["bb_mid"] and last["Close"] < last["bb_mid"] and last["Close"] < prev["Close"])

    @staticmethod
    def analyse_1_to_20(df):
        if df is None or len(df) < 3:
            return {"side": None, "hits": [], "bull_score": 0, "bear_score": 0}

        hits = []
        bull = 0
        bear = 0

        bd = RuleEngine.bullish_divergence(df)
        sd = RuleEngine.bearish_divergence(df)
        sq = RuleEngine.bollinger_squeeze(df)
        vol = RuleEngine.volume_surge(df)
        slope = RuleEngine.trend_slope(df)
        support = RuleEngine.support_rejection(df)
        resistance = RuleEngine.resistance_rejection(df)
        up = RuleEngine.breakout_up(df)
        down = RuleEngine.breakout_down(df)

        if bd:
            hits.append(1)
            bull += 16
        if sq:
            hits.append(2)
            bull += 6
            bear += 6
        if support:
            hits.append(3)
            bull += 14
        if vol:
            hits.append(4)
            bull += 8
            bear += 8
        if up:
            hits.append(5)
            bull += 18
        if slope > 0:
            hits.append(6)
            bull += 8
        if float(df.iloc[-1]["rsi"]) >= 55:
            hits.append(7)
            bull += 10
        if float(df.iloc[-1]["Close"]) > float(df.iloc[-1]["bb_upper"]):
            hits.append(8)
            bull += 12
        if float(df.iloc[-1]["Close"]) > float(df.iloc[-1]["bb_mid"]):
            hits.append(9)
            bull += 6
        if RuleEngine.candle_stats(df.iloc[-1])["bull"]:
            hits.append(10)
            bull += 6

        if sd:
            hits.append(11)
            bear += 16
        if sq:
            hits.append(12)
            bear += 6
        if resistance:
            hits.append(13)
            bear += 14
        if vol:
            hits.append(14)
            bear += 8
        if down:
            hits.append(15)
            bear += 18
        if slope < 0:
            hits.append(16)
            bear += 8
        if float(df.iloc[-1]["rsi"]) <= 45:
            hits.append(17)
            bear += 10
        if float(df.iloc[-1]["Close"]) < float(df.iloc[-1]["bb_lower"]):
            hits.append(18)
            bear += 12
        if float(df.iloc[-1]["Close"]) < float(df.iloc[-1]["bb_mid"]):
            hits.append(19)
            bear += 6
        if RuleEngine.candle_stats(df.iloc[-1])["bear"]:
            hits.append(20)
            bear += 6

        side = "bullish" if bull > bear else "bearish" if bear > bull else None
        return {"side": side, "hits": hits, "bull_score": bull, "bear_score": bear}

Overwriting /content/drive/MyDrive/MyTrade/analysis/rule_engine.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/analysis/ai_confirmation.py

from analysis.technical_analysis import TechnicalAnalysis
from analysis.rule_engine import RuleEngine

class AIConfirmation:
    @staticmethod
    def analyse_21(df, rule_result, phase="unknown"):
        if df is None or len(df) < 3:
            return {
                "signal": "None",
                "score": 0,
                "side": None,
                "reason": "",
                "confirmation_score": 0,
            }

        last = df.iloc[-1]
        prev = df.iloc[-2]
        stats = RuleEngine.candle_stats(last)

        bull_score = rule_result.get("bull_score", 0)
        bear_score = rule_result.get("bear_score", 0)
        side = rule_result.get("side")

        vol_ok = RuleEngine.volume_surge(df)
        sq_ok = RuleEngine.bollinger_squeeze(df)
        trend = RuleEngine.trend_slope(df)
        bd = TechnicalAnalysis.rsi_bullish_divergence(df)
        sd = TechnicalAnalysis.rsi_bearish_divergence(df)

        bull_confirm = 0
        bear_confirm = 0

        if side == "bullish":
            if vol_ok: bull_confirm += 14
            if sq_ok: bull_confirm += 8
            if trend > 0: bull_confirm += 10
            if float(last["Close"]) > float(prev["High"]): bull_confirm += 14
            if stats["lower_wick_ratio"] > stats["upper_wick_ratio"]: bull_confirm += 8
            if float(last["rsi"]) >= 55: bull_confirm += 10
            if bd: bull_confirm += 6
            if phase == "accumulation": bull_confirm += 8
            if sd: bull_confirm -= 12

        elif side == "bearish":
            if vol_ok: bear_confirm += 14
            if sq_ok: bear_confirm += 8
            if trend < 0: bear_confirm += 10
            if float(last["Close"]) < float(prev["Low"]): bear_confirm += 14
            if stats["upper_wick_ratio"] > stats["lower_wick_ratio"]: bear_confirm += 8
            if float(last["rsi"]) <= 45: bear_confirm += 10
            if sd: bear_confirm += 6
            if phase == "distribution": bear_confirm += 8
            if bd: bear_confirm -= 12

        confirmation_score = bull_confirm if side == "bullish" else bear_confirm if side == "bearish" else max(bull_confirm, bear_confirm)
        rule_score = max(bull_score, bear_score)
        final_score = rule_score + confirmation_score

        if side == "bullish" and confirmation_score >= 40 and rule_score >= 40:
            return {
                "signal": "Analyse 21 Bullish",
                "score": final_score,
                "side": "bullish",
                "reason": "rule and momentum confirmed",
                "confirmation_score": confirmation_score,
            }

        if side == "bearish" and confirmation_score >= 40 and rule_score >= 40:
            return {
                "signal": "Analyse 21 Bearish",
                "score": final_score,
                "side": "bearish",
                "reason": "rule and momentum confirmed",
                "confirmation_score": confirmation_score,
            }

        return {
            "signal": "None",
            "score": confirmation_score,
            "side": None,
            "reason": "confirmation weak",
            "confirmation_score": confirmation_score,
        }

Writing /content/drive/MyDrive/MyTrade/analysis/ai_confirmation.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/modules/telegram_alerts.py
import aiohttp

class TelegramAlerts:
    def __init__(self, app=None):
        self.app = app

    async def send_message(self, token, chat_id, text, parse_mode=None):
        if not token or not chat_id or not text:
            return {"ok": False, "description": "Missing token, chat_id, or text"}

        url = f"https://api.telegram.org/bot{token}/sendMessage"
        payload = {
            "chat_id": str(chat_id),
            "text": text,
        }
        if parse_mode:
            payload["parse_mode"] = parse_mode

        try:
            async with aiohttp.ClientSession() as session:
                async with session.post(url, json=payload, timeout=15) as resp:
                    data = await resp.json(content_type=None)
                    return data
        except Exception as e:
            return {"ok": False, "description": str(e)}

    async def test_connection(self, token, chat_id):
        result = await self.send_message(token, chat_id, "MyTradeAlert: Telegram connection test")
        if result.get("ok"):
            return True, "Telegram connected successfully"
        return False, result.get("description", "Telegram connection failed")

    def format_ai21_alert(self, symbol, side, price, tf, score, confirmation_score, phase, hits, reason, time_text):
        return (
            f"MyTradeAlert AI Confirmation\n"
            f"{symbol}\n"
            f"Analyse 21 {side}\n"
            f"Price: {price}\n"
            f"TF: {tf}\n"
            f"Score: {score}\n"
            f"AI Confirm: {confirmation_score}\n"
            f"Hits: {','.join(map(str, hits)) or '-'}\n"
            f"Phase: {phase}\n"
            f"Reason: {reason}\n"
            f"Time: {time_text}"
        )

Overwriting /content/drive/MyDrive/MyTrade/modules/telegram_alerts.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/modules/storage.py

import os
from kivy.storage.jsonstore import JsonStore

class AppStorage:
    def __init__(self, filename="data/persisted/app_store.json"):
        self.filename = filename
        self._ensure_parent()
        self.store = JsonStore(self.filename)

    def _ensure_parent(self):
        folder = os.path.dirname(self.filename)
        if folder and not os.path.exists(folder):
            os.makedirs(folder, exist_ok=True)

    def get(self, key, default=None):
        if self.store.exists(key):
            return self.store.get(key).get("value", default)
        return default

    def put(self, key, value):
        self.store.put(key, value=value)

    def exists(self, key):
        return self.store.exists(key)

    def delete(self, key):
        if self.store.exists(key):
            self.store.delete(key)

    def clear(self):
        for key in list(self.store.keys()):
            self.store.delete(key)

    def keys(self):
        return list(self.store.keys())

Overwriting /content/drive/MyDrive/MyTrade/modules/storage.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/modules/backtester.py
import pandas as pd
from analysis.technical_analysis import TechnicalAnalysis
from analysis.rule_engine import RuleEngine
from analysis.ai_confirmation import AIConfirmation

class Backtester:
    def __init__(self, app=None, data_fetcher=None, ai=None):
        self.app = app
        self.data_fetcher = data_fetcher
        self.ai = ai
        self.latest_results = []

    def _drawdown(self, equity):
        peak = equity.cummax()
        dd = (equity - peak) / peak.replace(0, pd.NA)
        return float(dd.min() * 100) if len(dd) else 0.0

    def _stats(self, trades, equity_curve):
        if not trades:
            return {'profit': 0.0, 'winrate': 0.0, 'loss': 0.0, 'total_trades': 0, 'max_drawdown': 0.0, 'score': 0}
        wins = [t for t in trades if t['pnl'] > 0]
        losses = [t for t in trades if t['pnl'] <= 0]
        profit = sum(t['pnl'] for t in trades)
        winrate = (len(wins) / len(trades)) * 100 if trades else 0.0
        loss = abs(sum(t['pnl'] for t in losses))
        dd = self._drawdown(pd.Series(equity_curve))
        score = int(max(0, profit + winrate - abs(dd)))
        return {'profit': round(profit, 2), 'winrate': round(winrate, 2), 'loss': round(loss, 2), 'total_trades': len(trades), 'max_drawdown': round(dd, 2), 'score': score}

    def _simulate(self, df):
        if df is None or len(df) < 40:
            return [], [100.0]
        trades, equity, in_pos, side, entry, entry_idx = [], [100.0], False, None, 0.0, 0
        for i in range(30, len(df) - 1):
            window = df.iloc[: i + 1].copy()
            rule = RuleEngine.analyse_1_to_20(window)
            phase = TechnicalAnalysis.market_phase(window)
            ai = AIConfirmation.analyse_21(window, rule, phase)
            if not in_pos and ai['signal'] != 'None':
                in_pos, side, entry, entry_idx = True, ai['side'], float(df['Close'].iloc[i + 1]), i + 1
                continue
            if in_pos:
                cur = float(df['Close'].iloc[i + 1])
                move = (cur - entry) / entry * 100
                pnl = move if side == 'bullish' else -move
                exit_now = False
                if side == 'bullish' and (cur < float(df['bb_mid'].iloc[i + 1]) or ai['signal'] == 'Analyse 21 Bearish'): exit_now = True
                if side == 'bearish' and (cur > float(df['bb_mid'].iloc[i + 1]) or ai['signal'] == 'Analyse 21 Bullish'): exit_now = True
                if exit_now or i + 1 == len(df) - 1:
                    trades.append({'entry_idx': entry_idx, 'exit_idx': i + 1, 'side': side, 'entry': entry, 'exit': cur, 'pnl': round(pnl, 2), 'score': ai.get('confirmation_score', 0), 'signal': ai.get('signal', 'None')})
                    equity.append(equity[-1] * (1 + pnl / 100))
                    in_pos, side = False, None
        return trades, equity

    async def backtest_symbol(self, symbol, timeframe='15m'):
        frames = await self.data_fetcher.prepare_indicator_frames(symbol, [timeframe])
        df = frames.get(timeframe)
        if df is None or df.empty:
            res = {'symbol': symbol, 'strategy': 'Analyse 21', 'profit': 0.0, 'winrate': 0.0, 'loss': 0.0, 'total_trades': 0, 'max_drawdown': 0.0, 'score': 0, 'confirmation_score': 0, 'phase': 'unknown'}
            self.latest_results.append(res); return res
        df = TechnicalAnalysis.add_indicators(df)
        trades, equity = self._simulate(df)
        stats = self._stats(trades, equity)
        last_rule = RuleEngine.analyse_1_to_20(df)
        last_phase = TechnicalAnalysis.market_phase(df)
        last_ai = AIConfirmation.analyse_21(df, last_rule, last_phase)
        res = {'symbol': symbol, 'strategy': 'Analyse 21', 'profit': stats['profit'], 'winrate': stats['winrate'], 'loss': stats['loss'], 'total_trades': stats['total_trades'], 'max_drawdown': stats['max_drawdown'], 'score': stats['score'], 'confirmation_score': last_ai.get('confirmation_score', 0), 'phase': last_phase}
        self.latest_results.append(res); return res

Overwriting /content/drive/MyDrive/MyTrade/modules/backtester.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/modules/ai_trend_learner.py

import asyncio
import threading
from datetime import datetime

from kivy.clock import Clock

from analysis.technical_analysis import TechnicalAnalysis
from analysis.rule_engine import RuleEngine
from analysis.ai_confirmation import AIConfirmation


class AITrendLearner:
    def __init__(self, app=None, data_fetcher=None, telegram=None):
        self.app = app
        self.data_fetcher = data_fetcher
        self.telegram = telegram
        self.running = False
        self.stop_flag = False
        self.latest_scan_results = []
        self.latest_symbol_states = {}
        self.last_error = ""

    def get_status_text(self):
        if self.running:
            return "AI Status: Running"
        return "AI Status: Stopped"

    def stop(self):
        self.stop_flag = True
        self.running = False

    def run_forever(self):
        if self.running:
            return
        self.running = True
        self.stop_flag = False
        try:
            asyncio.run(self._scan_loop())
        except Exception as e:
            self.last_error = str(e)
        finally:
            self.running = False

    async def _scan_loop(self):
        while not self.stop_flag:
            try:
                await self.scan_once()
            except Exception as e:
                self.last_error = str(e)
            await asyncio.sleep(10)

    async def scan_once(self):
        app = self.app
        watchlist = list(getattr(app, "watchlist", [])) if app else []
        if not watchlist:
            self.latest_scan_results = []
            return

        timeframes = ["1m", "3m", "5m", "10m", "15m", "30m", "1h"]
        results = []

        for symbol in watchlist:
            try:
                frames = await self.data_fetcher.prepare_indicator_frames(symbol, timeframes)
            except Exception as e:
                self.last_error = f"{symbol}: {e}"
                continue

            symbol_state = None

            for tf in timeframes:
                df = frames.get(tf)
                if df is None or len(df) < 3:
                    continue

                phase = TechnicalAnalysis.market_phase(df)
                rule_result = RuleEngine.analyse_1_to_20(df)
                ai21 = AIConfirmation.analyse_21(df, rule_result, phase)

                if ai21["signal"] != "None":
                    symbol_state = {
                        "symbol": symbol,
                        "signal": ai21["signal"],
                        "tf": tf,
                        "price": float(df["Close"].iloc[-1]),
                        "score": ai21.get("score", 0),
                        "confirmation_score": ai21.get("confirmation_score", 0),
                        "phase": phase,
                        "hits": rule_result.get("hits", []),
                        "side": ai21.get("side"),
                        "reason": ai21.get("reason", ""),
                        "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    }
                    results.append(symbol_state)
                    break

            if symbol_state is None:
                symbol_state = {
                    "symbol": symbol,
                    "signal": "None",
                    "tf": None,
                    "price": 0.0,
                    "score": 0,
                    "confirmation_score": 0,
                    "phase": "unknown",
                    "hits": [],
                    "side": None,
                    "reason": "",
                    "time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                }

            self.latest_symbol_states[symbol] = symbol_state

        self.latest_scan_results = results

        if app and hasattr(app, "telegram_enabled") and app.telegram_enabled:
            await self._send_alerts(results)

        Clock.schedule_once(lambda dt: self._refresh_ui())

    async def _send_alerts(self, results):
        app = self.app
        if not app:
            return

        token = getattr(app, "storage", None).get("tg_token", "") if hasattr(app, "storage") else ""
        chat_id = getattr(app, "storage", None).get("tg_chat_id", "") if hasattr(app, "storage") else ""

        if not token or not chat_id:
            return

        for item in results:
            try:
                message = self.telegram.format_ai21_alert(
                    item["symbol"],
                    item["signal"].replace("Analyse 21 ", ""),
                    item["price"],
                    item["tf"],
                    item["score"],
                    item["confirmation_score"],
                    item["phase"],
                    item["hits"],
                    item["reason"],
                    item["time"],
                )
                await self.telegram.send_message(token, chat_id, message)
            except Exception as e:
                self.last_error = str(e)

    def _refresh_ui(self):
        app = self.app
        if not app or not getattr(app, "root", None):
            return
        try:
            app.root.ids.scanner_screen.refresh_scanner(
                self.latest_scan_results,
                app.root.ids.scanner_screen.ids.active_only_switch.active,
            )
            app.root.ids.watchlist_screen.refresh_watchlist(
                getattr(app, "watchlist", []),
                self.latest_symbol_states,
            )
            app.root.ids.settings_screen.refresh_status(self.get_status_text(), self.last_error)
        except Exception as e:
            self.last_error = str(e)

Overwriting /content/drive/MyDrive/MyTrade/modules/ai_trend_learner.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/utils/stock_search.py

import asyncio

try:
    import yfinance as yf
except Exception:
    yf = None


class StockSearch:
    def __init__(self, app=None):
        self.app = app

    async def autocomplete(self, query, max_results=12):
        q = (query or "").strip()
        if len(q) < 1:
            return []

        if yf is None:
            return []

        def _search():
            try:
                if hasattr(yf, "Search"):
                    s = yf.Search(q, max_results=max_results)
                    quotes = getattr(s, "quotes", []) or []
                    out = []
                    for item in quotes[:max_results]:
                        sym = item.get("symbol", "")
                        name = item.get("shortname") or item.get("longname") or item.get("name") or ""
                        exch = item.get("exchange", "")
                        label = f"{sym} - {name}" if name else sym
                        if exch:
                            label = f"{label} ({exch})"
                        out.append(label)
                    return out
                return []
            except Exception:
                return []

        return await asyncio.to_thread(_search)

    async def validate_ticker(self, symbol):
        s = (symbol or "").strip().upper()
        if not s or yf is None:
            return False

        def _check():
            try:
                ticker = yf.Ticker(s)
                hist = ticker.history(period="7d", interval="1d")
                if hist is not None and len(hist) > 0:
                    return {
                        "symbol": s,
                        "valid": True,
                        "name": self._ticker_name(ticker),
                    }
                return False
            except Exception:
                return False

        result = await asyncio.to_thread(_check)
        return result

    def _ticker_name(self, ticker):
        try:
            info = ticker.fast_info if hasattr(ticker, "fast_info") else {}
        except Exception:
            pass

        try:
            info = ticker.info or {}
            return info.get("shortName") or info.get("longName") or info.get("displayName") or ""
        except Exception:
            return ""

    async def get_symbol_label(self, symbol):
        v = await self.validate_ticker(symbol)
        if not v:
            return symbol
        name = v.get("name", "")
        return f"{symbol} - {name}" if name else symbol

Overwriting /content/drive/MyDrive/MyTrade/utils/stock_search.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/utils/data_fetcher.py

import asyncio
from datetime import datetime, timedelta

try:
    import yfinance as yf
except Exception:
    yf = None

from analysis.technical_analysis import TechnicalAnalysis

class DataFetcher:
    def __init__(self, app=None):
        self.app = app

    async def fetch_ohlcv(self, symbol, timeframe="15m", period=None):
        if yf is None:
            return None

        symbol = (symbol or "").strip().upper()
        if not symbol:
            return None

        if period is None:
            period = self._default_period(timeframe)

        def _download():
            try:
                df = yf.download(
                    symbol,
                    period=period,
                    interval=timeframe,
                    auto_adjust=False,
                    progress=False,
                    threads=False,
                )
                if df is None or df.empty:
                    return None
                df = df.reset_index()
                df.columns = [str(c).title().replace("Datetime", "Date") for c in df.columns]
                return df
            except Exception:
                return None

        return await asyncio.to_thread(_download)

    async def prepare_indicator_frames(self, symbol, timeframes):
        frames = {}
        for tf in timeframes:
            raw = await self.fetch_ohlcv(symbol, tf)
            if raw is None or raw.empty:
                continue
            ind = TechnicalAnalysis.add_indicators(raw)
            if ind is not None and not ind.empty:
                frames[tf] = ind
        return frames

    def _default_period(self, timeframe):
        intraday = {
            "1m": "7d",
            "2m": "7d",
            "5m": "30d",
            "15m": "60d",
            "30m": "60d",
            "60m": "730d",
            "90m": "730d",
            "1h": "730d",
        }
        return intraday.get(timeframe, "2y")

    async def fetch_latest_price(self, symbol):
        df = await self.fetch_ohlcv(symbol, "1d", period="5d")
        if df is None or df.empty:
            return None
        return float(df["Close"].iloc[-1])

    async def fetch_multi_timeframe(self, symbol):
        timeframes = ["1m", "5m", "15m", "1h", "1d"]
        return await self.prepare_indicator_frames(symbol, timeframes)

Overwriting /content/drive/MyDrive/MyTrade/utils/data_fetcher.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/widgets/backtest_card.py
from kivy.metrics import dp
from kivymd.uix.card import MDCard
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel, MDIcon

class BacktestCard(MDCard):
    def __init__(self, result, **kwargs):
        super().__init__(**kwargs)
        self.result = result
        self.size_hint_y = None
        self.height = dp(160)
        self.padding = dp(12)
        self.radius = [16, 16, 16, 16]
        self.md_bg_color = (0.12, 0.12, 0.12, 1)

        row = MDBoxLayout(orientation="horizontal", spacing=dp(10))
        side = result.get("strategy", "Analyse 21")
        icon = "clipboard-text"
        color = (0.6, 0.7, 1, 1)

        row.add_widget(
            MDIcon(
                icon=icon,
                theme_text_color="Custom",
                text_color=color,
                size_hint_x=None,
                width=dp(28),
            )
        )

        col = MDBoxLayout(orientation="vertical", spacing=dp(4))
        col.add_widget(MDLabel(text=f"{result.get('symbol', '')} | {side}", bold=True))
        col.add_widget(
            MDLabel(
                text=f"Profit: {result.get('profit', 0)}%   Winrate: {result.get('winrate', 0)}%   Loss: {result.get('loss', 0)}%"
            )
        )
        col.add_widget(
            MDLabel(
                text=f"Trades: {result.get('total_trades', 0)}   DD: {result.get('max_drawdown', 0)}   Score: {result.get('score', 0)}"
            )
        )
        col.add_widget(
            MDLabel(
                text=f"AI Confirm: {result.get('confirmation_score', 0)}   Phase: {result.get('phase', 'unknown')}"
            )
        )

        row.add_widget(col)
        self.add_widget(row)

Overwriting /content/drive/MyDrive/MyTrade/widgets/backtest_card.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/widgets/watch_card.py
from kivy.metrics import dp
from kivymd.uix.card import MDCard
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel, MDIcon

class WatchCard(MDCard):
    def __init__(self, symbol, state, **kwargs):
        super().__init__(**kwargs)
        self.symbol = symbol
        self.state = state or {}

        signal = self.state.get("signal", "None")

        if signal in ("Buy Call", "Analyse 21 Bullish"):
            bg = (0.12, 0.18, 0.14, 1)
            icon = "arrow-up-bold"
            icon_color = (0.35, 0.9, 0.45, 1)
        elif signal in ("Buy Put", "Analyse 21 Bearish"):
            bg = (0.18, 0.12, 0.12, 1)
            icon = "arrow-down-bold"
            icon_color = (0.95, 0.35, 0.35, 1)
        else:
            bg = (0.11, 0.11, 0.11, 1)
            icon = "eye-outline"
            icon_color = (0.7, 0.7, 0.7, 1)

        self.size_hint_y = None
        self.height = dp(190)
        self.padding = dp(12)
        self.radius = [16, 16, 16, 16]
        self.md_bg_color = bg

        row = MDBoxLayout(orientation="horizontal", spacing=dp(10))

        row.add_widget(
            MDIcon(
                icon=icon,
                theme_text_color="Custom",
                text_color=icon_color,
                size_hint_x=None,
                width=dp(28),
            )
        )

        col = MDBoxLayout(orientation="vertical", spacing=dp(4))
        col.add_widget(MDLabel(text=symbol, bold=True))
        col.add_widget(MDLabel(text=f"Signal: {state.get('signal', 'None')}"))
        col.add_widget(MDLabel(text=f"TF: {state.get('tf', '')}   Price: {state.get('price', 0)}"))
        col.add_widget(MDLabel(text=f"Score: {state.get('score', 0)}   AI Confirm: {state.get('confirmation_score', 0)}"))
        col.add_widget(MDLabel(text=f"Hits: {','.join(map(str, state.get('hits', []))) or '-'}"))
        col.add_widget(MDLabel(text=f"Reason: {state.get('reason', '')}"))
        col.add_widget(MDLabel(text=f"Phase: {state.get('phase', 'unknown')}"))

        row.add_widget(col)
        self.add_widget(row)

Overwriting /content/drive/MyDrive/MyTrade/widgets/watch_card.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/widgets/signal_card.py
from kivy.metrics import dp
from kivymd.uix.card import MDCard
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel, MDIcon

class SignalCard(MDCard):
    def __init__(self, item, **kwargs):
        super().__init__(**kwargs)
        item = item or {}

        signal = item.get("signal", "None")
        if signal in ("Buy Call", "Analyse 21 Bullish"):
            bg = (0.12, 0.18, 0.14, 1)
            icon = "arrow-up-bold"
            icon_color = (0.35, 0.9, 0.45, 1)
        elif signal in ("Buy Put", "Analyse 21 Bearish"):
            bg = (0.18, 0.12, 0.12, 1)
            icon = "arrow-down-bold"
            icon_color = (0.95, 0.35, 0.35, 1)
        else:
            bg = (0.12, 0.12, 0.12, 1)
            icon = "chart-line"
            icon_color = (0.7, 0.7, 0.7, 1)

        self.size_hint_y = None
        self.height = dp(128)
        self.padding = dp(12)
        self.radius = [16, 16, 16, 16]
        self.md_bg_color = bg

        row = MDBoxLayout(orientation="horizontal", spacing=dp(10))
        row.add_widget(
            MDIcon(
                icon=icon,
                theme_text_color="Custom",
                text_color=icon_color,
                size_hint_x=None,
                width=dp(28),
            )
        )

        col = MDBoxLayout(orientation="vertical", spacing=dp(3))
        col.add_widget(MDLabel(text=f"{item.get('symbol', '')} | {item.get('tf', '')}", bold=True))
        col.add_widget(MDLabel(text=f"Signal: {signal}   Price: {item.get('price', '')}   Score: {item.get('score', 0)}"))
        col.add_widget(MDLabel(text=f"AI Confirm: {item.get('confirmation_score', 0)}   Phase: {item.get('phase', 'unknown')}"))
        col.add_widget(MDLabel(text=f"Hits: {','.join(map(str, item.get('hits', []))) or '-'}"))
        col.add_widget(MDLabel(text=f"Reason: {item.get('reason', '')}"))
        col.add_widget(MDLabel(text=f"Time: {item.get('time', '')}"))

        row.add_widget(col)
        self.add_widget(row)

Overwriting /content/drive/MyDrive/MyTrade/widgets/signal_card.py


In [ ]:
import sys
project_root = '/content/drive/MyDrive/MyTrade'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

try:
    from widgets import SignalCard, WatchCard, BacktestCard
    print("✅ Widgets package verified successfully with KivyMD 1.2.0 compatibility.")
except ImportError as e:
    print(f"❌ Widgets verification failed: {e}")

✅ Widgets package verified successfully with KivyMD 1.2.0 compatibility.


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/modules/__init__.py

from .ai_trend_learner import AITrendLearner
from .backtester import Backtester
from .storage import AppStorage
from .telegram_alerts import TelegramAlerts

__all__ = [
    "AITrendLearner",
        "Backtester",
            "AppStorage",
                "TelegramAlerts",
                ]

Overwriting /content/drive/MyDrive/MyTrade/modules/__init__.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/analysis/__init__.py

from .technical_analysis import TechnicalAnalysis
from .rule_engine import RuleEngine
from .ai_confirmation import AIConfirmation

__all__ = [
    "TechnicalAnalysis",
        "RuleEngine",
            "AIConfirmation",
            ]

Overwriting /content/drive/MyDrive/MyTrade/analysis/__init__.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/widgets/__init__.py

from .signal_card import SignalCard
from .watch_card import WatchCard
from .backtest_card import BacktestCard

__all__ = ["SignalCard", "WatchCard", "BacktestCard"]

Overwriting /content/drive/MyDrive/MyTrade/widgets/__init__.py


In [ ]:
import sys
project_root = '/content/drive/MyDrive/MyTrade'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

try:
    from widgets import SignalCard, WatchCard, BacktestCard
    print("✅ Widgets package verified successfully with KivyMD 1.2.0 compatibility.")
except ImportError as e:
    print(f"❌ Widgets verification failed: {e}")

✅ Widgets package verified successfully with KivyMD 1.2.0 compatibility.


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/utils/__init__.py

from .data_fetcher import DataFetcher
from .stock_search import StockSearch
from modules.storage import AppStorage

__all__ = ["DataFetcher", "StockSearch", "AppStorage"]

Overwriting /content/drive/MyDrive/MyTrade/utils/__init__.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/screens/__init__.py

from .scanner_screen import ScannerScreen
from .watchlist_screen import WatchlistScreen
from .backtest_screen import BacktestScreen
from .settings_screen import SettingsScreen

__all__ = [
    "ScannerScreen",
        "WatchlistScreen",
            "BacktestScreen",
                "SettingsScreen",
                ]

Overwriting /content/drive/MyDrive/MyTrade/screens/__init__.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/utils/__init__.py

from .data_fetcher import DataFetcher
from .stock_search import StockSearch
from modules.storage import AppStorage

__all__ = ["DataFetcher", "StockSearch", "AppStorage"]

Overwriting /content/drive/MyDrive/MyTrade/utils/__init__.py


In [ ]:
%%writefile /content/drive/MyDrive/MyTrade/buildozer.spec
[app]
title = MyTradeAlert
package.name = mytradealert
package.domain = org.mytradealert

source.dir = .
source.include_exts = py,kv,png,jpg,jpeg,gif,webp,json,ttf,otf,txt
version = 0.1

# Removed fixed version for numpy to avoid git checkout errors
requirements = python3,kivy==2.3.1,kivymd==1.2.0,aiohttp==3.11.11,yfinance==0.2.51,pandas==2.2.3,numpy,pandas-ta

orientation = portrait
fullscreen = 0

android.api = 33
android.minapi = 24
android.ndk = 25b
android.ndk_api = 24

android.permissions = INTERNET,WRITE_EXTERNAL_STORAGE,READ_EXTERNAL_STORAGE

source.exclude_dirs = .git,__pycache__,build,dist,.idea,.vscode,tests
source.exclude_patterns = *.pyc,*.pyo,*~

[buildozer]
log_level = 2
warn_on_root = 1

[buildozer:android]
android.archs = arm64-v8a,armeabi-v7a

Overwriting /content/drive/MyDrive/MyTrade/buildozer.spec


In [ ]:
import os

# Path to your spec file
spec_path = '/content/drive/MyDrive/MyTrade/buildozer.spec'

if os.path.exists(spec_path):
    with open(spec_path, 'r') as f:
        content = f.read()

    # Fix numpy version to 1.26.4 for compatibility with p4a
    new_requirements = 'requirements = python3,kivy==2.3.1,kivymd==1.2.0,aiohttp==3.11.11,yfinance==0.2.51,pandas==2.2.3,numpy==1.26.4,pandas-ta'

    import re
    content = re.sub(r'requirements = .*', new_requirements, content)

    with open(spec_path, 'w') as f:
        f.write(content)
    print('✅ buildozer.spec updated: numpy pinned to 1.26.4')
else:
    print('❌ buildozer.spec not found.')

✅ buildozer.spec updated: numpy pinned to 1.26.4


In [ ]:
# @title 🚀 Re-run Build with Numpy Fix
import os

project_path = '/content/drive/MyDrive/MyTrade'
local_path = '/root/build_project'

# Clean local build dir to avoid cached numpy failures
if os.path.exists(local_path):
    import shutil
    shutil.rmtree(local_path)

# Re-copy and run
shutil.copytree(project_path, local_path, ignore=shutil.ignore_patterns('.buildozer', 'bin', '.git'))
%cd {local_path}

print('Starting build with pinned numpy...')
!yes | /usr/local/bin/buildozer -v android debug

In [ ]:
! python /content/drive/MyDrive/MyTrade/widgets/watch_card.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/widgets/signal_card.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/widgets/backtest_card.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/stock_search.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/memory_manager.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/data_fetcher.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/watchlist_screen.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/settings_screen.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/scanner_screen.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/backtest_screen.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/telegram_alerts.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/storage.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/backtester.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/ai_trend_learner.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/analysis/technical_analysis.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/analysis/rule_engine.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/analysis/ai_confirmation.py

In [ ]:
%cd /root/MyTrade_Local
# Set environment variables to ensure clean logging and non-interactive mode
os.environ['TERM'] = 'dumb'
os.environ['PYTHONUNBUFFERED'] = '1'

print('🚀 Starting clean local build...')
!yes | buildozer -v android debug 2>&1 | grep --line-buffered -v 'stty: '

In [ ]:
!export HOME=/root
!export BUILDozer_DIR=/root/.buildozer

In [ ]:
!export P4A_HOME=/root/.buildozer
!export BUILDozer_DIR=/root/.buildozer

In [ ]:
!echo $HOME
!echo $P4A_HOME
!pwd
!ls -ld /root/.buildozer
!ls -ld /content/drive/MyDrive/MyTrade

In [ ]:
!cd /content/drive/MyDrive/MyTrade
!rm -rf .buildozer
!rm -rf /root/.buildozer
!find /content/drive/MyDrive/MyTrade -name ".git" -type d -prune -exec rm -f {}/shallow.lock \;

In [ ]:
%cd /content/drive/MyDrive/MyTrade
!rm -rf .buildozer
!rm -rf /root/.buildozer
!mkdir -p /root/.buildozer

In [ ]:
!rm -rf /root/.buildozer/android/platform/build-*/build/other_builds/libffi

In [ ]:
!pwd
!ls -la

In [ ]:
!sudo apt-get update
!sudo apt-get install -y git zip unzip openjdk-17-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev build-essential patchelf
!pip install --upgrade pip
!pip install buildozer cython==0.29.33

In [ ]:
%cd /root
# Fixed the 'Install' typo and ensured all build-essential tools for p4a are present
!sudo apt install -y build-essential libffi-dev libssl-dev python3-dev zip unzip openjdk-17-jdk xclip gettext libtool autoconf autogen

!sudo apt update
!sudo apt install -y xclip gettext libtool autoconf autogen

!sudo apt Update
!sudo apt install -y autopoint gettext

!sudo apt update
!sudo apt install -y patchelf

!sudo apt update

!pip install --upgrade buildozer cython==0.29.33



In [ ]:
%cd /content/drive/MyDrive/MyTrade
# Automatically accept root warnings and build using the clean storage directory
!yes | buildozer android p4a -- --help
!yes | buildozer -v android debug --storage-dir /root/.buildozer